# 09 – Ablation study

**Goal (proposal Section 3.5):** remove each component one at a time and measure the effect. If performance drops without it, the component is earning its place.

| Component | Tested on |
|---|---|
| Platt calibration | Framingham (Kaggle) |
| Patient-similarity graph | Framingham (Kaggle) |
| Exam history (LSTM) | Framingham longitudinal |
| Federation | UCI, 4 hospitals |

## Step 1 – Setup

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"; os.environ["OMP_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings("ignore")
from xgboost import XGBClassifier
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, pyreadr
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.neighbors import kneighbors_graph
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import GATConv

ablation = []   # results collected here

## Step 2 – Ablation 1: Platt calibration

Same XGBoost model, with vs without Platt scaling. Metric: **Brier score** (lower = better).

In [2]:
df = pd.read_csv("../data/framingham.csv")
df.loc[(df.currentSmoker == 0) & (df.cigsPerDay.isna()), "cigsPerDay"] = 0
df = df.drop(columns=["education"])
X = df.drop(columns=["TenYearCHD"]); y = df["TenYearCHD"]
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

imp = SimpleImputer(strategy="median").fit(X_train)
w = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=w, random_state=42, n_jobs=1).fit(imp.transform(X_train), y_train)

to_logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))).reshape(-1, 1)
p_raw = xgb.predict_proba(imp.transform(X_test))[:, 1]
platt = LogisticRegression().fit(to_logit(xgb.predict_proba(imp.transform(X_cal))[:, 1]), y_cal)
p_cal = platt.predict_proba(to_logit(p_raw))[:, 1]

ablation.append({"component": "Platt calibration", "dataset": "Framingham", "metric": "Brier (lower=better)",
                 "with": brier_score_loss(y_test, p_cal), "without": brier_score_loss(y_test, p_raw)})
ablation[-1]

{'component': 'Platt calibration',
 'dataset': 'Framingham',
 'metric': 'Brier (lower=better)',
 'with': 0.12064354121685028,
 'without': 0.19374464452266693}

## Step 3 – Ablation 2: patient-similarity graph

Same GAT model and training, **with** the 10-nearest-neighbour graph vs **without** any links.

In [3]:
torch.manual_seed(42); np.random.seed(42)
sc = StandardScaler().fit(imp.transform(X_train))
X_all = pd.concat([X_train, X_cal, X_test]); y_all = pd.concat([y_train, y_cal, y_test])
Z = sc.transform(imp.transform(X_all)).astype("float32")
n, n_tr, n_cal = len(Z), len(X_train), len(X_cal)
train_mask = torch.zeros(n, dtype=torch.bool); train_mask[:n_tr] = True
cal_mask = torch.zeros(n, dtype=torch.bool); cal_mask[n_tr:n_tr+n_cal] = True
test_mask = torch.zeros(n, dtype=torch.bool); test_mask[n_tr+n_cal:] = True

A = kneighbors_graph(Z, n_neighbors=10, include_self=False).tocoo()
knn_edges = to_undirected(torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long))
no_edges = torch.empty((2, 0), dtype=torch.long)
x_t = torch.tensor(Z); y_t = torch.tensor(y_all.values, dtype=torch.float)
pos_weight = torch.tensor(w, dtype=torch.float)

class GAT(nn.Module):
    def __init__(self, d, h=32):
        super().__init__()
        self.c1 = GATConv(d, h, heads=4, dropout=0.3); self.c2 = GATConv(h * 4, 1, dropout=0.3)
    def forward(self, x, e):
        x = F.dropout(x, 0.3, self.training); x = F.elu(self.c1(x, e)); x = F.dropout(x, 0.3, self.training)
        return self.c2(x, e).squeeze(-1)

def train_gat(edges, seed=42):
    torch.manual_seed(seed)
    m = GAT(Z.shape[1]); opt = torch.optim.Adam(m.parameters(), lr=0.005, weight_decay=5e-4)
    best, state = 0, None
    for _ in range(300):
        m.train(); opt.zero_grad()
        F.binary_cross_entropy_with_logits(m(x_t, edges)[train_mask], y_t[train_mask],
                                           pos_weight=pos_weight).backward(); opt.step()
        m.eval()
        with torch.no_grad(): a = roc_auc_score(y_t[cal_mask], m(x_t, edges)[cal_mask])
        if a > best: best, state = a, {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(state); m.eval()
    with torch.no_grad(): return roc_auc_score(y_t[test_mask], m(x_t, edges)[test_mask])

ablation.append({"component": "Patient-similarity graph (GAT)", "dataset": "Framingham", "metric": "AUC",
                 "with": train_gat(knn_edges), "without": train_gat(no_edges)})
ablation[-1]

{'component': 'Patient-similarity graph (GAT)',
 'dataset': 'Framingham',
 'metric': 'AUC',
 'with': 0.6883699367122726,
 'without': 0.6960571853672737}

## Step 4 – Ablation 3: exam history (LSTM)

Same LSTM, trained on the **3-exam sequence** vs **only the last exam**.

In [4]:
lf = pyreadr.read_r("../data/framingham.rda")["framingham"]
TEN = 3652
FEATS = ["SEX", "AGE", "SYSBP", "DIABP", "TOTCHOL", "CURSMOKE", "CIGPDAY",
         "BMI", "DIABETES", "BPMEDS", "HEARTRTE", "GLUCOSE"]

ne = lf.groupby("RANDID")["PERIOD"].nunique()
d = lf[lf["RANDID"].isin(ne[ne == 3].index)].sort_values(["RANDID", "PERIOD"])
e3 = d[d["PERIOD"] == 3].set_index("RANDID")
e3 = e3[~((e3["CVD"] == 1) & (e3["TIMECVD"] <= e3["TIME"])) & (e3["PREVCHD"] == 0) & (e3["PREVSTRK"] == 0)]
ids = e3.index
yl = ((e3["CVD"] == 1) & (e3["TIMECVD"] <= e3["TIME"] + TEN)).astype(int)
d = d[d["RANDID"].isin(ids)].copy()

tr_ids, tmp = train_test_split(ids, test_size=0.4, stratify=yl.loc[ids], random_state=42)
ca_ids, te_ids = train_test_split(tmp, test_size=0.5, stratify=yl.loc[tmp], random_state=42)
rows_tr = d[d["RANDID"].isin(tr_ids)]
imp_l = SimpleImputer(strategy="median").fit(rows_tr[FEATS])
sc_l = StandardScaler().fit(imp_l.transform(rows_tr[FEATS]))
d[FEATS] = sc_l.transform(imp_l.transform(d[FEATS]))
d["GAP"] = (d.groupby("RANDID")["TIME"].diff().fillna(0) / 365.25 - 3) / 3
SF = FEATS + ["GAP"]

byp = {pid: g[SF].values for pid, g in d.groupby("RANDID")}
seq = lambda L: torch.tensor(np.stack([byp[i] for i in L]), dtype=torch.float32)
Xs = {"tr": seq(tr_ids), "ca": seq(ca_ids), "te": seq(te_ids)}
ys = {k: torch.tensor(yl.loc[v].values, dtype=torch.float32)
      for k, v in [("tr", tr_ids), ("ca", ca_ids), ("te", te_ids)]}

class RiskLSTM(nn.Module):
    def __init__(self, d, h=32):
        super().__init__()
        self.lstm = nn.LSTM(d, h, batch_first=True); self.drop = nn.Dropout(0.3); self.out = nn.Linear(h, 1)
    def forward(self, x):
        h, _ = self.lstm(x); return self.out(self.drop(h[:, -1])).squeeze(-1)

def train_lstm(n_exams, seed=42):
    torch.manual_seed(seed)
    sl = lambda X: X[:, -n_exams:, :]          # keep only the last n exams
    m = RiskLSTM(len(SF)); opt = torch.optim.Adam(m.parameters(), lr=0.005, weight_decay=1e-4)
    best, state = 0, None
    for _ in range(200):
        m.train(); opt.zero_grad()
        F.binary_cross_entropy_with_logits(m(sl(Xs["tr"])), ys["tr"]).backward(); opt.step()
        m.eval()
        with torch.no_grad(): a = roc_auc_score(ys["ca"], m(sl(Xs["ca"])))
        if a > best: best, state = a, {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(state); m.eval()
    with torch.no_grad(): return roc_auc_score(ys["te"], m(sl(Xs["te"])))

ablation.append({"component": "Exam history (LSTM: 3 exams vs last only)", "dataset": "Framingham longitudinal",
                 "metric": "AUC", "with": train_lstm(3), "without": train_lstm(1)})
ablation[-1]

{'component': 'Exam history (LSTM: 3 exams vs last only)',
 'dataset': 'Framingham longitudinal',
 'metric': 'AUC',
 'with': 0.7606484424666243,
 'without': 0.7201472769654587}

## Step 5 – Ablation 4: federation

Federated logistic regression (FedAvg, 4 hospitals) vs each hospital training **alone**.
Compared per hospital, then averaged over Cleveland, Hungary and VA.
Switzerland excluded: only 2 healthy patients in its test set.

In [5]:
cols = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach","exang","oldpeak","slope","ca","thal","num"]
fr = []
for name, f in {"Cleveland": "processed.cleveland.data", "Hungary": "processed.hungarian.data",
                "Switzerland": "processed.switzerland.data", "VA": "processed.va.data"}.items():
    t = pd.read_csv(f"../data/UCI/{f}", header=None, names=cols, na_values="?"); t["hospital"] = name; fr.append(t)
u = pd.concat(fr, ignore_index=True); u["target"] = (u["num"] > 0).astype(int)
u = u.drop(columns=["num", "slope", "ca", "thal"])
u.loc[u["chol"] == 0, "chol"] = np.nan; u.loc[u["trestbps"] == 0, "trestbps"] = np.nan
u = pd.get_dummies(u, columns=["cp"], prefix="cp", dtype=int)
uf = [c for c in u.columns if c not in ("hospital", "target")]

hosp = {h: train_test_split(g[uf], g["target"], test_size=0.3, stratify=g["target"], random_state=42)
        for h, g in u.groupby("hospital", sort=False)}
Xa = pd.concat([v[0] for v in hosp.values()])
ui = SimpleImputer(strategy="median").fit(Xa); us = StandardScaler().fit(ui.transform(Xa))
prep = lambda Xd: us.transform(ui.transform(Xd))
sig = lambda z: 1 / (1 + np.exp(-z))

def local_train(wv, b, Xd, yd, epochs=5, lr=0.1, l2=0.01):
    for _ in range(epochs):
        e = sig(Xd @ wv + b) - yd
        wv = wv - lr * (Xd.T @ e / len(yd) + l2 * wv); b = b - lr * e.mean()
    return wv, b

# Federated (FedAvg, 50 rounds)
clients = {h: (prep(v[0]), v[2].values) for h, v in hosp.items()}
N = sum(len(c[1]) for c in clients.values())
wv, b = np.zeros(len(uf)), 0.0
for _ in range(50):
    ups = [local_train(wv.copy(), b, Xd, yd) + (len(yd),) for Xd, yd in clients.values()]
    wv = sum(n_ * wi for wi, bi, n_ in ups) / N; b = sum(n_ * bi for wi, bi, n_ in ups) / N

Xte_all = np.vstack([prep(v[1]) for v in hosp.values()])
yte_all = np.concatenate([v[3].values for v in hosp.values()])
p_fed = sig(Xte_all @ wv + b)
# Local only: each hospital trains its own model
p_loc = np.concatenate([LogisticRegression(max_iter=2000).fit(prep(v[0]), v[2]).predict_proba(prep(v[1]))[:, 1]
                        for v in hosp.values()])

off = np.cumsum([0] + [len(v[3]) for v in hosp.values()])
per_h = {}
for k, h in enumerate(hosp):
    s_ = slice(off[k], off[k + 1])
    per_h[h] = (roc_auc_score(yte_all[s_], p_fed[s_]), roc_auc_score(yte_all[s_], p_loc[s_]))
print(pd.DataFrame(per_h, index=["federated", "local_only"]).T.round(3))

main = ["Cleveland", "Hungary", "VA"]
ablation.append({"component": "Federation (FedAvg vs each hospital alone)", "dataset": "UCI – mean of 3 hospitals",
                 "metric": "AUC", "with": np.mean([per_h[h][0] for h in main]),
                 "without": np.mean([per_h[h][1] for h in main])})
ablation.append({"component": "Federation – VA hospital only", "dataset": "UCI – VA",
                 "metric": "AUC", "with": per_h["VA"][0], "without": per_h["VA"][1]})

             federated  local_only
Cleveland        0.897       0.892
Hungary          0.890       0.912
Switzerland      0.400       0.457
VA               0.785       0.723


## Step 6 – Ablation summary

In [6]:
tab = pd.DataFrame(ablation)
tab["change"] = tab["with"] - tab["without"]
tab.round(3)

,component,dataset,metric,with,without,change
0,Platt calibration,Framingham,Brier (lower=better),0.121,0.194,-0.073
1,Patient-similarity graph (GAT),Framingham,AUC,0.688,0.696,-0.008
2,Exam history (LSTM: 3 exams vs last only),Framingham longitudinal,AUC,0.761,0.720,0.041
3,Federation (FedAvg vs each hospital alone),UCI – mean of 3 hospitals,AUC,0.858,0.842,0.015
4,Federation – VA hospital only,UCI – VA,AUC,0.785,0.723,0.062


## Summary

| Component | Dataset | Metric | With | Without | Effect |
|---|---|---|---|---|---|
| Platt calibration | Framingham | Brier ↓ | 0.121 | 0.195 | Large improvement |
| Exam history (LSTM) | Framingham longitudinal | AUC | 0.761 | 0.720 | +0.041 |
| Federation | UCI (mean of 3 hospitals) | AUC | 0.858 | 0.842 | +0.015 |
| Federation – VA only | UCI | AUC | 0.785 | 0.723 | +0.062 |
| Patient-similarity graph | Framingham | AUC | 0.688 | 0.696 | −0.008 |

- **Calibration, exam history and federation each improve results** – removing them makes the model worse
- **Federation helps the weakest hospital most** (VA +0.062)
- **The graph does not improve accuracy**; its contribution is case-based explainability (similar patients)
- Federation compared by **per-hospital AUC**, since pooling predictions from separately trained hospital models inflates AUC through differences in hospital disease rates